# Lab 1 - Qiskit Patterns, the four steps

Every Qiskit program that touches real hardware has the same four steps in it.
IBM calls the shape a **Qiskit pattern**, and once you can see it, the library
stops looking like a pile of unrelated classes.

```
   map  ->  optimize  ->  execute  ->  post-process
```

The exercises drilled each piece separately. This lab is where they line up.

Nothing here is graded. Change the numbers, break things on purpose, rerun the
cell. Everything runs on a local simulator, so no account and no network are
needed, and nothing you do here can spend a QPU minute.

## Why four and not one

You have a question. The machine has gates. Those are not the same language, and
three separate translations sit between them:

1. **Map** turns the question into a circuit, and into the observables you want
   measured. This is the only step that is about your problem.
2. **Optimize** rewrites the circuit into gates this particular device physically
   has, on qubits that are physically connected. Skip it and hardware refuses the
   job outright.
3. **Execute** runs it, through a Sampler if you want bitstrings or an Estimator
   if you want a number.
4. **Post-process** turns whatever came back into an answer to the original
   question.

Steps 2 and 3 are where beginners lose days, because 0.x tutorials fold them into
a single `execute()` call that no longer exists.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# A snapshot of a real 5-qubit device: real gate set, real connectivity, real
# error rates. Everything below is aimed at this, without leaving the laptop.
backend = FakeManilaV2()
print(backend.name, "-", backend.num_qubits, "qubits")
print("gates it actually has:", sorted(backend.target.operation_names))

## Step 1 - Map

The question here: **are these two qubits correlated?**

Mapping it means two things, and they are separate objects. A circuit that
prepares the state, and an observable that asks the question. `ZZ` is +1 when the
two qubits agree and -1 when they disagree, so its expectation value is exactly
"how correlated are they".

In [ ]:
def bell_circuit():
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    return qc


circuit = bell_circuit()
observable = SparsePauliOp("ZZ")

print(circuit.draw())
print("observable:", observable)

Note what the circuit does **not** have: a measurement. An observable says what
to measure, so adding `measure_all()` here would be asking twice.

That is the first thing to try changing. Replace `cx` with nothing and rerun the
whole notebook: the correlation collapses to 0.

## Step 2 - Optimize

The circuit above uses `h` and `cx`. Look at the gate list printed earlier: this
device has no `h`. It has `rz`, `sx`, `x`, and `cx`.

Transpiling is not an optimisation you can skip for a small circuit. It is the
translation into gates that physically exist, onto qubits that are physically
wired together. Hardware rejects anything else.

In [ ]:
pass_manager = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pass_manager.run(circuit)

print("before:", dict(circuit.count_ops()), " depth", circuit.depth())
print("after :", dict(isa_circuit.count_ops()), " depth", isa_circuit.depth())
print()
print(isa_circuit.draw(idle_wires=False))

One `h` became two `rz` and an `sx`, and the depth went from 2 to 4. On real
hardware deeper means more time for the state to decay, which is lab 2.

An observable has to be translated too, because the transpiler is free to move
your logical qubits onto different physical ones. `apply_layout` follows that
move. Forgetting it is a classic silent wrong answer: no error, just a number
that measures the wrong qubits.

In [ ]:
isa_observable = observable.apply_layout(isa_circuit.layout)
print("logical :", observable.paulis)
print("physical:", isa_observable.paulis)
print()
print("The physical one is padded out to", isa_observable.num_qubits, "qubits,")
print("because the device has that many and the layout chose which to use.")

## Step 3 - Execute

Two primitives, two questions.

- **Sampler**: run it and give me the bitstrings. Needs a measurement in the
  circuit.
- **Estimator**: run it and give me `<observable>`. Needs an observable, and no
  measurement.

The local versions used here are `StatevectorSampler` and `StatevectorEstimator`.
Against IBM hardware you would swap in `SamplerV2` and `EstimatorV2` from
`qiskit_ibm_runtime`. The call shape is deliberately identical.

In [ ]:
# Estimator: one number, straight out.
estimator_result = StatevectorEstimator().run([(circuit, observable)]).result()
correlation = float(estimator_result[0].data.evs)

# Sampler: bitstrings, so this copy needs a measurement.
measured = circuit.copy()
measured.measure_all()
sampler_result = StatevectorSampler(seed=1).run([measured], shots=4096).result()
counts = sampler_result[0].data.meas.get_counts()

print(f"Estimator -> {correlation:+.4f}   (raw: {correlation!r})")
print("Sampler   ->", dict(sorted(counts.items())))

## Step 4 - Post-process

The Estimator already answered the question, which is the point of choosing it.
The Sampler handed back raw tallies, so the question has to be reconstructed:
`ZZ` is +1 on `00` and `11`, and -1 on `01` and `10`.

Doing it both ways is the exercise. They agree, and seeing them agree is what
makes the Estimator feel less like magic.

In [ ]:
def zz_from_counts(counts):
    shots = sum(counts.values())
    agree = counts.get("00", 0) + counts.get("11", 0)
    disagree = counts.get("01", 0) + counts.get("10", 0)
    return (agree - disagree) / shots


print(f"from the Estimator : {correlation:+.4f}")
print(f"from the counts    : {zz_from_counts(counts):+.4f}")
print()
print("Same number. The Estimator did this arithmetic for you, and on hardware it")
print("would also have split the observable into runnable measurements first.")

## The whole pattern, once

Written out with nothing between the steps, it is short. Short enough that it is
worth keeping this cell as the template you copy when you start something new.

In [ ]:
def run_pattern(circuit, observable, backend):
    # 1. map: handed in, because mapping is the part that is about your problem
    # 2. optimize
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa = pm.run(circuit)
    isa_obs = observable.apply_layout(isa.layout)
    # 3. execute
    result = StatevectorEstimator().run([(isa, isa_obs)]).result()
    # 4. post-process
    return float(result[0].data.evs)


def h_only():
    qc = QuantumCircuit(2)
    qc.h(0)
    return qc


for label, build_it in [("entangled (h + cx)", bell_circuit), ("no cx at all", h_only)]:
    value = run_pattern(build_it(), SparsePauliOp("ZZ"), backend)
    print(f"{label:20} <ZZ> = {value:+.4f}")

`<ZZ> = 1` for the Bell pair: perfectly correlated. `<ZZ> = 0` for a lone
Hadamard on qubit 0: qubit 1 never learned anything, so the two agree exactly
half the time.

## What changes on hardware

Almost nothing, which is the reason the pattern is worth learning as a pattern.

```python
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
# steps 2 and 4 are identical, step 3 becomes:
result = EstimatorV2(mode=backend).run([(isa, isa_obs)]).result()
```

Three lines differ. The mapping, the transpilation and the post-processing are
the same code.

The one thing that does change is the answer: it stops being exact. That is lab
2, and it is not a bug.

In [ ]:
from quantum_exercises.backends import get_backend

# The same selection exercise 13 makes: a real QPU when one is reachable,
# otherwise a simulator carrying a hardware noise model.
selection = get_backend(min_num_qubits=2)
print("would run on:", selection.describe())
print("because     :", selection.reason)